# Lab02 MLP: controlled from-scratch ablations

Suite round2, paired seeds 42/43, 600 epochs, source images RGB 64x64. No pretrained weights. Select the configuration by mean validation macro-F1 before test. The test split has been inspected in prior project experiments. The embedded source snapshot, rather than a mutable Git branch, defines this run.


In [ ]:
import base64, hashlib, io, json, os, subprocess, sys, time, zipfile
from pathlib import Path
import torch

SOURCE_SHA256 = "68736a9735f2f4928181ee0593686c3db33402b2146e85b93ac768d605f3ffd2"
SOURCE_BASE_REVISION = "87f1611ce7c0ce3f710fa24d885e7207c991bca7"
payload = base64.b64decode("...")
assert hashlib.sha256(payload).hexdigest() == SOURCE_SHA256
root = Path("/tmp/wastelens-mlp-snapshot")
root.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(root)
run_root = Path("/kaggle/working/wastelens-mlp-round2")
run_root.mkdir(exist_ok=False)
(run_root / "source-snapshot.zip").write_bytes(payload)
(run_root / "source-snapshot.json").write_text(json.dumps({
    "sha256": SOURCE_SHA256, "base_revision": SOURCE_BASE_REVISION,
    "note": "Embedded local source including the new MLP experiments; base revision alone does not define this snapshot."
}, indent=2), encoding="utf-8")
assert torch.cuda.device_count() >= 2, "Requires Kaggle T4 x2"
visible = os.environ.get("CUDA_VISIBLE_DEVICES")
slots = visible.split(",") if visible else [str(i) for i in range(torch.cuda.device_count())]
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(2)], flush=True)
subprocess.run([sys.executable, str(root / "scripts/prepare_trashnet.py")], cwd=root, check=True)


In [ ]:
from contextlib import ExitStack
workers = []
with ExitStack() as stack:
    for seed, slot in zip((42, 43), slots[:2], strict=True):
        log_path = run_root / f"seed-{seed}.log"
        log = stack.enter_context(log_path.open("w", encoding="utf-8"))
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=slot, PYTHONPATH=str(root / "src"),
                   OMP_NUM_THREADS="2", CUBLAS_WORKSPACE_CONFIG=":4096:8")
        args = [sys.executable, "-u", "-m", "wastelens.mlp_experiments", "train",
                "--output", str(run_root / f"seed-{seed}"), "--seed", str(seed),
                "--epochs", "600", "--batch-size", "64", "--device", "cuda", "--suite", "round2"]
        worker = subprocess.Popen(args, cwd=root, env=env, stdout=log, stderr=subprocess.STDOUT)
        workers.append((seed, worker, log_path))
        print(f"Started seed {seed} on slot {slot}", flush=True)
    previous = {}
    while any(worker.poll() is None for _, worker, _ in workers):
        for seed, _, log_path in workers:
            lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
            if lines and lines[-1] != previous.get(seed):
                print(f"Seed {seed}: {lines[-1]}", flush=True)
                previous[seed] = lines[-1]
        time.sleep(10)
    codes = [worker.wait() for _, worker, _ in workers]
for (seed, _, log_path), exit_code in zip(workers, codes, strict=True):
    print(log_path.read_text(encoding="utf-8", errors="replace")[-1800:])
    if exit_code:
        raise RuntimeError(f"Seed {seed} failed with exit code {exit_code}")
env = dict(os.environ, PYTHONPATH=str(root / "src"), CUBLAS_WORKSPACE_CONFIG=":4096:8")
subprocess.run([sys.executable, "-u", "-m", "wastelens.mlp_experiments", "summarize",
                "--output", str(run_root), "--seeds", "42", "43", "--device", "cuda", "--suite", "round2"],
               cwd=root, env=env, check=True)


In [ ]:
import csv
import matplotlib.pyplot as plt
summary = json.loads((run_root / "run-summary.json").read_text(encoding="utf-8"))
rows = summary["validation"]
fig, ax = plt.subplots(figsize=(11, 4.5), layout="constrained")
ax.bar([row["name"] for row in rows], [100 * row["val_accuracy_mean"] for row in rows],
       yerr=[100 * row["val_accuracy_std"] for row in rows], capsize=4, color="#267b69")
ax.set(ylabel="Validation accuracy (%)", title="From-scratch MLP: mean and sample SD over two seeds")
ax.tick_params(axis="x", rotation=30)
fig.savefig(run_root / "validation-accuracy.png", dpi=170)
plt.show()
print("Winner selected on validation:", summary["winner"])
for name, result in summary["test"].items():
    print(name, "test accuracy", result["accuracy_mean"], "macro-F1", result["macro_f1_mean"])
